# 🌊 PINNs-MVP: Kolmogorov Flow 完整實驗指南
## Physics-Informed Neural Networks for 2D Turbulent Flow Reconstruction

---

**最後更新**: 2025-11-27  
**版本**: v3.1 (Causal Training Edition)  
**新功能**: ✅ Causal Training | ✅ Curriculum Learning | ✅ Re=100 kf=4 配置

---

### 📚 本 Notebook 內容概覽

#### Part 1: 專案介紹與環境設定
- 1.1 Kolmogorov Flow 簡介
- 1.2 Reynolds 數定義與驗證 ⭐
- 1.3 環境檢查與 GPU 加速設定

#### Part 2: DNS 資料生成
- 2.1 生成 Kolmogorov Flow DNS
- 2.2 Reynolds 數計算與驗證
- 2.3 DNS 結果視覺化

#### Part 3: QR-Pivot 感測器配置
- 3.1 生成 K=100 最優感測點
- 3.2 感測器品質分析
- 3.3 視覺化感測點分佈

#### Part 4: 模型訓練
- 4.1 配置文件準備
- 4.2 Colab 訓練（T4/A100 GPU）
- 4.3 切換到 A100 GPU ⭐
- 4.4 訓練監控與檢查點

#### Part 5: 評估與診斷
- 5.1 物理一致性驗證
- 5.2 場重建誤差分析
- 5.3 訓練失敗診斷

#### Part 6: 進階實驗
- 6.1 多雷諾數實驗（Re=56, Re=158, Re=197）
- 6.2 K-scan 實驗（K=50, 100, 200）
- 6.3 課程學習策略

---

### 🎯 學習目標

完成本 Notebook 後，您將能夠：
- ✅ 生成並驗證 Kolmogorov Flow DNS 資料
- ✅ 正確計算並驗證 Reynolds 數（Musacchio & Boffetta 2014 定義）
- ✅ 使用 QR-Pivot 選擇最優感測點
- ✅ 在多種硬體上訓練 PINNs（MPS/CUDA/CPU）
- ✅ 診斷訓練失敗並優化超參數

---

### ⚙️ Google Colab 硬體選項

| GPU 類型 | 訓練時間（3000 epochs） | 訂閱需求 | 設定 |
|---------|------------------------|---------|------|
| **NVIDIA A100** | 4-6 小時 ⭐ | Colab Pro+ | 自動偵測 |
| **NVIDIA V100** | 8-10 小時 | Colab Pro | 自動偵測 |
| **NVIDIA T4** | 12-16 小時 | 免費/Pro | 自動偵測 |

**推薦**: A100 GPU（Colab Pro+ 訂閱）以獲得最快訓練速度

---

## Part 0: Google Colab 初始化

**⚠️ 重要**：本 Notebook 專為 **Google Colab (T4/A100 GPU)** 設計，不適合本地端運行。  
請在執行任何實驗前先完成以下初始化步驟。

In [1]:
# 0.1 檢測是否在 Colab 環境中
try:
    import google.colab
    IN_COLAB = True
    print("✅ 檢測到 Google Colab 環境")
except ImportError:
    IN_COLAB = False
    print("⚠️  警告：本 Notebook 設計用於 Google Colab")
    print("   本地端訓練請使用: scripts/train.py")

⚠️  警告：本 Notebook 設計用於 Google Colab
   本地端訓練請使用: scripts/train.py


In [ ]:
# 0.2 掛載 Google Drive
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive 已掛載至 /content/drive")
else:
    raise RuntimeError(
        "❌ 錯誤：本 Notebook 必須在 Google Colab 上運行\n"
        "   請前往: https://colab.research.google.com\n"
        "   或使用本地端訓練: python scripts/train.py --cfg <config.yml>"
    )

In [ ]:
# 0.3 切換到專案目錄
import os

# 專案路徑（請根據您的 Drive 結構修改）
PROJECT_PATH = '/content/drive/MyDrive/pinns-mvp'

# 檢查專案是否存在
if os.path.exists(PROJECT_PATH):
    os.chdir(PROJECT_PATH)
    print(f"✅ 已切換到專案目錄: {os.getcwd()}")
else:
    print(f"❌ 專案目錄不存在: {PROJECT_PATH}")
    print("\n請確認：")
    print("  1. 是否已將專案上傳至 Google Drive")
    print("  2. 路徑是否正確（區分大小寫）")
    print("\n📂 您的 Drive 根目錄內容：")
    !ls /content/drive/MyDrive/
    print("\n💡 提示：參考 docs/COLAB_SETUP_GUIDE.md")

In [ ]:
# 0.4 驗證專案結構
required_dirs = ['configs', 'scripts', 'pinnx', 'data']
missing_dirs = [d for d in required_dirs if not os.path.exists(d)]

if missing_dirs:
    print(f"❌ 錯誤：缺少以下目錄: {missing_dirs}")
    print("\n可能原因：")
    print("  1. 專案未完整上傳至 Google Drive")
    print("  2. 路徑錯誤（請檢查 Cell 0.3 的 PROJECT_PATH）")
    print("\n解決方案：")
    print("  - 重新上傳完整專案資料夾")
    print("  - 或參考 docs/COLAB_SETUP_GUIDE.md")
else:
    print("✅ 專案結構完整，可以開始實驗！")
    print("\n📂 專案目錄結構：")
    !ls -lh

### 📝 Colab 設定說明

**如果上述步驟失敗，請手動設定**：

1. **上傳專案到 Google Drive**：
   ```bash
   # 本地端執行
   cd pinns-mvp
   zip -r pinns-mvp.zip .
   # 上傳 pinns-mvp.zip 到 Google Drive
   # 在 Drive 中解壓縮
   ```

2. **修改專案路徑**（如果路徑不同）：
   - 在上方 Cell 0.3 中修改 `PROJECT_PATH` 變數
   - 例如：`PROJECT_PATH = '/content/drive/MyDrive/研究/pinns-mvp'`

3. **確認 GPU 加速**：
   - 點擊 **執行階段 → 變更執行階段類型**
   - 硬體加速器選擇 **GPU** (T4 或 A100)
   - 點擊儲存

---

---

## Part 1: 專案介紹與環境設定

### 1.1 Kolmogorov Flow 簡介

**Kolmogorov Flow** 是一個 2D 週期性流動，由正弦強迫驅動：

$$
\begin{cases}
\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} + v \frac{\partial u}{\partial y} = -\frac{\partial p}{\partial x} + \nu \nabla^2 u + A \sin(k_f y) \\
\frac{\partial v}{\partial t} + u \frac{\partial v}{\partial x} + v \frac{\partial v}{\partial y} = -\frac{\partial p}{\partial y} + \nu \nabla^2 v \\
\frac{\partial u}{\partial x} + \frac{\partial v}{\partial y} = 0
\end{cases}
$$

其中：
- $A$: 強迫振幅（forcing amplitude）
- $k_f$: 強迫波數（forcing wavenumber）
- $\nu$: 動力黏度（kinematic viscosity）

**物理特性**：
- 在適當 Reynolds 數下，流動從層流轉變為湍流
- 出現大尺度渦結構與能量逆級串（inverse energy cascade）
- 適合測試 PINNs 在湍流重建中的能力

### 1.2 Reynolds 數定義與驗證 ⭐

**重要**: 本專案採用 **Musacchio & Boffetta (2014)** 的 Reynolds 數定義：

$$
\text{Re} = \frac{\sqrt{f_0} \times L^{3/2}}{\nu} = \frac{\sqrt{A} \times (2\pi/k_f)^{3/2}}{\nu}
$$

其中：
- $f_0 = A$: 強迫振幅
- $L = 2\pi/k_f$: 強迫波長（特徵長度）

**為什麼需要驗證？**
- DNS 檔案名稱可能標示錯誤的 Re 值
- 配置文件中的 `nu`、`k_f`、`Re` 必須一致
- 錯誤的 Re 會導致物理不一致，訓練失敗

**驗證工具**: `scripts/calculate_reynolds_parameters.py`

In [ ]:
# 1.2.1 驗證 Reynolds 數
!python scripts/calculate_reynolds_parameters.py --f0 1.0 --nu 0.0125 --k 8

# 預期輸出：Re = 55.68
# 流動狀態：過渡/弱湍流

In [ ]:
# 1.2.2 驗證 Re=100 kf=4 配置（本 Notebook 使用的配置）
!python scripts/calculate_reynolds_parameters.py --f0 1.0 --nu 0.019687 --k 4

# ✅ 預期輸出：Re = 100.0
# 流動狀態：湍流

### 1.3 環境檢查與 GPU 加速設定

In [ ]:
# 1.3.1 檢查 PyTorch 與 GPU
import torch
import sys

print(f"Python 版本: {sys.version}")
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
print(f"MPS 可用: {torch.backends.mps.is_available()}")

if torch.cuda.is_available():
    print(f"GPU 設備: {torch.cuda.get_device_name(0)}")
    print(f"GPU 記憶體: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    print("使用 Apple Silicon GPU (MPS)")
else:
    print("⚠️ 警告: 僅 CPU 可用，訓練將非常緩慢")

In [ ]:
# 1.3.2 自動選擇最佳設備
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ 使用 CUDA GPU 加速")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ 使用 MPS GPU 加速（Apple Silicon）")
else:
    device = torch.device('cpu')
    print("⚠️ 使用 CPU（訓練緩慢）")

print(f"設備: {device}")

---

## Part 2: DNS 資料生成

### 2.1 生成 Kolmogorov Flow DNS

In [ ]:
# 2.1.1 生成 Re=100 的 DNS 資料（k_f=4, ν=0.019687）
!python scripts/generate_kolmogorov_dns.py \
  --N 512 --nu 0.019687 --k_f 4 --T_end 40.0 \
  --output data/kolmogorov_dns_re100_512x512_kf4.h5

# 參數說明：
# --N 512: 網格解析度 512x512
# --nu 0.019687: 動力黏度（Re=100）
# --k_f 4: 強迫波數（較低波數，流型較簡單）
# --T_end 40.0: 模擬時間

### 2.2 Reynolds 數計算與驗證 ⚠️

In [ ]:
# 2.2.1 驗證剛生成的 DNS 檔案的實際雷諾數
!python scripts/calculate_reynolds_parameters.py --f0 1.0 --nu 0.019687 --k 4

# ✅ 預期: Re = 100.0
# ❌ 如果不是 100.0，請檢查參數設定！

In [ ]:
# 2.2.2 檢查 DNS 檔案內的實際參數
import h5py

with h5py.File('data/kolmogorov_dns_re100_512x512_kf4.h5', 'r') as f:
    print("DNS 檔案參數：")
    print(f"  ν (nu): {f['config'].attrs['nu']}")
    print(f"  k_f: {f['config'].attrs['k_f']}")
    print(f"  A (forcing amplitude): {f['config'].attrs.get('forcing_amplitude', 1.0)}")
    print(f"  網格大小: {f['u'].shape}")

# ✅ 確認 nu=0.019687, k_f=4, A=1.0
# 使用這些參數計算 Re，確保與配置文件一致！

### 2.3 DNS 結果視覺化

In [ ]:
# 2.3.1 視覺化 DNS 結果（速度場、渦度、能譜）
# 注意：visualize_kolmogorov_results.py 僅支援訓練結果 .npz 檔案
# DNS .h5 檔案視覺化使用手動 matplotlib 程式碼（見下方範例）

import matplotlib.pyplot as plt
import h5py
import numpy as np
import os

os.makedirs('results/dns_analysis_re100', exist_ok=True)

# 載入 DNS 資料
with h5py.File('data/kolmogorov_dns_re100_512x512_kf4.h5', 'r') as f:
    u = f['u'][-1, :, :]  # 最後時間步
    v = f['v'][-1, :, :]
    x = np.linspace(0, 2*np.pi, u.shape[1])
    y = np.linspace(0, 2*np.pi, u.shape[0])
    X, Y = np.meshgrid(x, y)

# 繪製速度場
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
speed = np.sqrt(u**2 + v**2)
vorticity = np.gradient(v, axis=1) - np.gradient(u, axis=0)

im1 = ax1.contourf(X, Y, speed, levels=50, cmap='viridis')
ax1.set_title('Speed Field (DNS)', fontsize=14)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
plt.colorbar(im1, ax=ax1, label='|u|')

im2 = ax2.contourf(X, Y, vorticity, levels=50, cmap='RdBu_r')
ax2.set_title('Vorticity Field (DNS)', fontsize=14)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
plt.colorbar(im2, ax=ax2, label='ω')

plt.tight_layout()
plt.savefig('results/dns_analysis_re100/velocity_field_final.png', dpi=150)
plt.show()

print('✅ DNS 視覺化完成: results/dns_analysis_re100/')

---

## Part 3: QR-Pivot 感測器配置

### 3.1 生成 K=100 最優感測點

In [ ]:
# 3.1.1 使用 DEIM 演算法生成 K=100 感測點（Re=100 kf=4）
!python scripts/generate_sensors_k500.py \
  --input data/kolmogorov_dns_re100_512x512_kf4.h5 \
  --K 100 --n-modes 50 \
  --output data/kolmogorov_qr_sensors_re100_kf4_K100.npz

# 參數說明：
# --K 100: 感測器數量
# --n-modes 50: POD 模態數量
# 輸出：kolmogorov_qr_sensors_re100_kf4_K100.npz（包含座標與速度值）

### 3.2 感測器品質分析

In [ ]:
# 3.2.1 視覺化感測點分佈與品質指標（Re=100 kf=4）
!python scripts/visualize_qr_sensors.py \
  --input data/kolmogorov_qr_sensors_re100_kf4_K100.npz \
  --output results/sensor_analysis_re100_kf4_K100/

# 生成圖表：
# - sensor_distribution_2d.png: 2D 分佈圖
# - sensor_quality_metrics.png: 品質指標（條件數、能量比例）
# - sensor_statistics.png: 統計分析

In [ ]:
# 3.2.2 檢查感測器品質指標
# 注意：感測器檔案包含以下 keys:
#   - sensor_x, sensor_y: 感測點座標 (非 'coords')
#   - sensor_u, sensor_v: 感測點速度值
#   - K: 感測器數量
#   - condition_number, energy_ratio: 品質指標
import numpy as np

sensors = np.load('data/kolmogorov_qr_sensors_re100_kf4_K100.npz')

print("📊 感測器配置摘要:")
print(f"  總數: {sensors['K']}")
print(f"  空間座標: x ∈ [{sensors['sensor_x'].min():.2f}, {sensors['sensor_x'].max():.2f}]")
print(f"           y ∈ [{sensors['sensor_y'].min():.2f}, {sensors['sensor_y'].max():.2f}]")

if 'condition_number' in sensors:
    cond = sensors['condition_number']
    print(f"\n🎯 品質指標:")
    print(f"  條件數: {cond:.2f}")
    if cond < 50:
        print("  ✅ 優秀（< 50）")
    elif cond < 100:
        print("  ✅ 良好（50-100）")
    else:
        print("  ⚠️ 警告（> 100），可能導致數值不穩定")

if 'energy_ratio' in sensors:
    energy = sensors['energy_ratio']
    print(f"  能量比例: {energy:.4f}")
    if energy > 0.95:
        print("  ✅ 優秀（> 0.95）")
    elif energy > 0.90:
        print("  ✅ 良好（0.90-0.95）")
    else:
        print("  ⚠️ 警告（< 0.90），可能無法捕捉高頻特徵")

### 3.3 視覺化感測點分佈

In [ ]:
# 3.3.1 繪製感測點分佈（疊加在速度場上）
import matplotlib.pyplot as plt
import h5py

# 載入 DNS 資料
with h5py.File('data/kolmogorov_dns_re100_512x512_kf4.h5', 'r') as f:
    u = f['u'][-1, :, :]  # 最後時間步的 u 場
    v = f['v'][-1, :, :]
    x = np.linspace(0, 2*np.pi, u.shape[1])
    y = np.linspace(0, 2*np.pi, u.shape[0])
    X, Y = np.meshgrid(x, y)

# 載入感測點
sensors = np.load('data/kolmogorov_qr_sensors_re100_kf4_K100.npz')
sensor_x = sensors['sensor_x']  # 修正：使用 'sensor_x' 而非 'coords'
sensor_y = sensors['sensor_y']  # 修正：使用 'sensor_y'

# 繪圖
fig, ax = plt.subplots(figsize=(10, 10))
speed = np.sqrt(u**2 + v**2)
im = ax.contourf(X, Y, speed, levels=50, cmap='viridis')
ax.scatter(sensor_x, sensor_y,
           c='red', s=20, marker='x', label=f'Sensors (K={len(sensor_x)})')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('QR-Pivot 感測點分佈（疊加在速度場上）')
ax.legend()
plt.colorbar(im, ax=ax, label='Speed')
plt.tight_layout()
plt.savefig('results/sensor_distribution_overlay.png', dpi=150)
plt.show()

print("✅ 感測點分佈圖已儲存至 results/sensor_distribution_overlay.png")

---

## Part 4: 模型訓練

### 4.1 配置文件準備

In [ ]:
# 4.1.1 檢查配置文件是否存在
import os

config_file = 'configs/kolmogorov_re100_kf4_K100.yml'

if os.path.exists(config_file):
    print(f"✅ 配置文件存在: {config_file}")
    !cat {config_file} | head -50
else:
    print(f"❌ 配置文件不存在: {config_file}")
    print("請檢查 configs/ 目錄")

In [ ]:
# 4.1.2 驗證配置文件中的 Reynolds 數
import yaml

with open('configs/kolmogorov_re100_kf4_K100.yml', 'r') as f:
    config = yaml.safe_load(f)

nu = config['physics']['nu']
kf = config['data']['kolmogorov_config']['physics_params']['k_f']
Re_config = config['data']['kolmogorov_config']['physics_params']['Re']

print("📋 配置文件參數:")
print(f"  ν (nu): {nu}")
print(f"  k_f: {kf}")
print(f"  Re (宣稱): {Re_config}")

# 計算實際 Re
f0 = 1.0  # 假設 forcing amplitude = 1.0
L = 2 * np.pi / kf
Re_actual = np.sqrt(f0) * L**(3/2) / nu

print(f"\n🧮 實際計算 Re: {Re_actual:.2f}")

if abs(Re_actual - Re_config) < 1.0:
    print("✅ Re 一致！可以開始訓練")
else:
    print(f"❌ Re 不一致！差異: {abs(Re_actual - Re_config):.2f}")
    print("請修正配置文件或重新生成 DNS 資料")

### 4.2 Colab 訓練（T4/A100 GPU）

In [ ]:
# 4.2.1 啟動訓練（Colab GPU）
# 新功能：✅ Causal Training + ✅ Curriculum Learning
# 訓練時間：T4 ~10-12 小時 | A100 ~3-4 小時（4000 epochs）

!python scripts/train.py \
  --cfg configs/kolmogorov_re100_kf4_K100.yml

# 檢查點將保存至: checkpoints/kolmogorov_re100_kf4_K100/
# 日誌保存至: log/kolmogorov_re100_kf4_K100.log

### 4.2.5 🆕 Causal Training 與 Curriculum Learning 說明

**本配置文件（v3.0）新增兩大訓練策略**：

#### 📌 1. Causal Training（時間因果權重）

**原理**：早期時間點應優先滿足物理約束，因為它們是後續時間的「因」。

$$
w(t) = \exp\left(-\epsilon \int_0^t \text{Loss}(\tau) d\tau\right)
$$

**配置參數**：
```yaml
causal_weighting: true
causal_eps: 0.5        # 因果約束強度（0.5 = 中等）
causal_n_bins: 10      # 時間分窗數量
```

**實際行為**（時間範圍 [20, 40]）：
- **早期時間** (t=20-25): w ≈ 1.0（最大權重，優先學習）
- **中期時間** (t=30-35): w ≈ 0.6-0.8
- **後期時間** (t=35-40): w ≈ 0.3-0.5

**預期效果**：
- ✅ 加速收斂（減少前期震盪）
- ✅ 提升時間一致性（避免後期時間違反物理）
- ✅ 與 DNS 時間演化更吻合

#### 📌 2. Curriculum Learning（課程學習）

**策略**：從簡單流動（低 Re 層流）逐步過渡到複雜流動（高 Re 湍流）。

**5 階段訓練流程**：

| 階段 | Re | ν | Epochs | 流動狀態 | 目標 L2 |
|------|-----|---------|---------|---------|--------|
| **Stage 1** | 10 | 0.196870 | 0-300 | 層流 | 30% |
| **Stage 2** | 30 | 0.065623 | 300-800 | 過渡流 | 25% |
| **Stage 3** | 50 | 0.039374 | 800-1500 | 弱湍流 | 20% |
| **Stage 4** | 80 | 0.024609 | 1500-2500 | 湍流 | 18% |
| **Stage 5** | 100 | 0.019687 | 2500-4000 | 目標湍流 | **15%** |

**學習率遞減**：0.001 → 0.0001（隨階段降低）

**PDE 權重遞增**：2.0 → 6.0（逐漸強化物理約束）

**預期效果**：
- ✅ 避免訓練初期就陷入湍流複雜結構
- ✅ 逐步建立物理直覺（層流 → 湍流）
- ✅ 提升最終收斂精度

### 4.3 切換到 A100 GPU（Colab Pro/Pro+）⭐

In [ ]:
# 4.3.1 檢查當前 GPU 類型
!nvidia-smi --query-gpu=name,memory.total --format=csv

# 如何切換到 A100：
# 1. 點擊右上角「執行階段 → 變更執行階段類型」
# 2. 硬體加速器：GPU
# 3. GPU 類型：A100（需 Colab Pro+ 訂閱）
# 4. 點擊「儲存」並重新執行 Part 0 初始化

### 4.4 訓練監控與檢查點

In [ ]:
# 4.4.1 檢查訓練進度
!tail -50 log/kolmogorov_re100_kf4_K100.log

In [ ]:
# 4.4.2 列出已保存的檢查點
checkpoint_dir = 'checkpoints/kolmogorov_re100_kf4_K100/'

if os.path.exists(checkpoint_dir):
    checkpoints = sorted(os.listdir(checkpoint_dir))
    print(f"📂 檢查點目錄: {checkpoint_dir}")
    print(f"✅ 找到 {len(checkpoints)} 個檢查點:\n")
    for ckpt in checkpoints:
        path = os.path.join(checkpoint_dir, ckpt)
        size = os.path.getsize(path) / 1e6  # MB
        print(f"  {ckpt} ({size:.1f} MB)")
else:
    print(f"⚠️ 檢查點目錄不存在: {checkpoint_dir}")
    print("訓練可能尚未開始或未保存檢查點")

---

## Part 5: 評估與診斷

### 5.1 物理一致性驗證

In [ ]:
# 5.1.1 使用檢查點評估物理一致性
!python scripts/evaluate_checkpoint.py \
  --checkpoint checkpoints/kolmogorov_re100_kf4_K100/best_model.pth \
  --config configs/kolmogorov_re100_kf4_K100.yml

# 輸出指標：
# - 連續性方程殘差（divergence）
# - 動量方程殘差（momentum_x, momentum_y）
# - 邊界條件誤差

### 5.2 場重建誤差分析

In [ ]:
# 5.2.1 評估場重建精度（相對 L2 誤差）
!python scripts/evaluate_kolmogorov_full.py \
  --checkpoint checkpoints/kolmogorov_re100_kf4_K100/best_model.pth \
  --dns-file data/kolmogorov_dns_re100_512x512_kf4.h5 \
  --output results/evaluation_re100_kf4/

# 生成指標：
# - u, v, p 的相對 L2 誤差
# - 能譜比較
# - 統計量對比（均值、標準差）

In [ ]:
# 5.2.2 快速評估與視覺化（Kolmogorov Flow）
!python scripts/evaluate_kolmogorov_quick.py \
  --checkpoint checkpoints/kolmogorov_re100_kf4_K100/best_model.pth \
  --config configs/kolmogorov_re100_kf4_K100.yml \
  --output results/evaluation_re100_kf4/ \
  --n-points 256

# 生成：
# - evaluation_results.npz: 預測場數據
# - fields_visualization.png: u, v, p, vorticity, divergence
# - loss_history.png: 訓練損失曲線

In [ ]:
# 5.2.3 生成 PINNs vs DNS vs Error 三面板對比圖 ⭐
import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import zoom
import os

print('🎨 生成 PINNs vs DNS 對比圖...')
print('=' * 70)

# 1. 載入 DNS 真值（時間平均）
dns_file = 'data/kolmogorov_dns_re100_512x512_kf4.h5'
with h5py.File(dns_file, 'r') as f:
    # 取時間窗口 [20, 40] 的平均（穩態）
    time = f['time'][:]
    t_start_idx = np.argmin(np.abs(time - 20.0))
    t_end_idx = np.argmin(np.abs(time - 40.0))

    u_dns_mean = f['u'][t_start_idx:t_end_idx, :, :].mean(axis=0)
    v_dns_mean = f['v'][t_start_idx:t_end_idx, :, :].mean(axis=0)
    p_dns_mean = f['p'][t_start_idx:t_end_idx, :, :].mean(axis=0)

print(f'✅ DNS 真值載入完成（時間平均 t ∈ [20, 40]）')
print(f'   DNS 網格大小: {u_dns_mean.shape}')

# 2. 載入 PINNs 預測
results_file = 'results/evaluation_re100_kf4/evaluation_results.npz'
if not os.path.exists(results_file):
    print(f'❌ 錯誤：找不到 {results_file}')
    print('   請先執行上一個 Cell (5.2.2)')
else:
    results = np.load(results_file)
    u_pinn = results['u']
    v_pinn = results['v']
    p_pinn = results['p']
    X = results['X']
    Y = results['Y']

    print(f'✅ PINNs 預測載入完成')
    print(f'   PINNs 網格大小: {u_pinn.shape}')
    print()

    # 3. Resize DNS 到相同解析度
    if u_dns_mean.shape != u_pinn.shape:
        zoom_factors = (u_pinn.shape[0] / u_dns_mean.shape[0],
                       u_pinn.shape[1] / u_dns_mean.shape[1])
        u_dns = zoom(u_dns_mean, zoom_factors, order=3)
        v_dns = zoom(v_dns_mean, zoom_factors, order=3)
        p_dns = zoom(p_dns_mean, zoom_factors, order=3)
        print(f'✅ DNS 場已插值至 PINNs 網格大小: {u_dns.shape}')
    else:
        u_dns = u_dns_mean
        v_dns = v_dns_mean
        p_dns = p_dns_mean

    # 4. 計算誤差與統計
    u_error = np.abs(u_pinn - u_dns)
    v_error = np.abs(v_pinn - v_dns)
    p_error = np.abs(p_pinn - p_dns)

    u_l2 = np.linalg.norm(u_error) / np.linalg.norm(u_dns) * 100
    v_l2 = np.linalg.norm(v_error) / np.linalg.norm(v_dns) * 100
    p_l2 = np.linalg.norm(p_error) / np.linalg.norm(p_dns) * 100

    print('📊 相對 L2 誤差:')
    print(f'   u: {u_l2:.2f}%')
    print(f'   v: {v_l2:.2f}%')
    print(f'   p: {p_l2:.2f}%')
    print()

    # 5. 繪製三面板對比圖
    fig, axes = plt.subplots(3, 3, figsize=(20, 18))

    fields = [
        ('u (x-velocity)', u_pinn, u_dns, u_error, 'RdBu_r'),
        ('v (y-velocity)', v_pinn, v_dns, v_error, 'RdBu_r'),
        ('p (pressure)', p_pinn, p_dns, p_error, 'viridis')
    ]

    for i, (name, pinn, dns, error, cmap) in enumerate(fields):
        # 計算共享的 colorbar 範圍
        vmin = min(pinn.min(), dns.min())
        vmax = max(pinn.max(), dns.max())

        # PINNs 預測
        im0 = axes[i, 0].contourf(X, Y, pinn, levels=50, cmap=cmap, vmin=vmin, vmax=vmax)
        axes[i, 0].set_title(f'{name} - PINNs Prediction', fontsize=14, fontweight='bold')
        axes[i, 0].set_xlabel('x', fontsize=12)
        axes[i, 0].set_ylabel('y', fontsize=12)
        axes[i, 0].set_aspect('equal')
        cbar0 = plt.colorbar(im0, ax=axes[i, 0])
        cbar0.ax.tick_params(labelsize=10)

        # DNS 真值
        im1 = axes[i, 1].contourf(X, Y, dns, levels=50, cmap=cmap, vmin=vmin, vmax=vmax)
        axes[i, 1].set_title(f'{name} - DNS Ground Truth', fontsize=14, fontweight='bold')
        axes[i, 1].set_xlabel('x', fontsize=12)
        axes[i, 1].set_ylabel('y', fontsize=12)
        axes[i, 1].set_aspect('equal')
        cbar1 = plt.colorbar(im1, ax=axes[i, 1])
        cbar1.ax.tick_params(labelsize=10)

        # 絕對誤差
        im2 = axes[i, 2].contourf(X, Y, error, levels=50, cmap='hot')
        rel_l2 = [u_l2, v_l2, p_l2][i]
        axes[i, 2].set_title(f'{name} - Absolute Error (L2: {rel_l2:.2f}%)',
                            fontsize=14, fontweight='bold')
        axes[i, 2].set_xlabel('x', fontsize=12)
        axes[i, 2].set_ylabel('y', fontsize=12)
        axes[i, 2].set_aspect('equal')
        cbar2 = plt.colorbar(im2, ax=axes[i, 2])
        cbar2.ax.tick_params(labelsize=10)

    plt.suptitle('PINNs vs DNS Comparison (Kolmogorov Flow Re=100)',
                 fontsize=18, fontweight='bold', y=0.995)
    plt.tight_layout()

    # 儲存
    output_file = 'results/evaluation_re100_kf4/pinns_vs_dns_comparison.png'
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    plt.savefig(output_file, dpi=200, bbox_inches='tight')
    print(f'✅ 對比圖已保存: {output_file}')
    plt.show()

    # 6. 總結
    print()
    print('=' * 70)
    print('🏆 評估總結')
    print('=' * 70)
    print(f'  相對 L2 誤差（目標 < 15%）：')
    print(f'    u: {u_l2:.2f}% {"✅" if u_l2 < 15 else "❌"}')
    print(f'    v: {v_l2:.2f}% {"✅" if v_l2 < 15 else "❌"}')
    print(f'    p: {p_l2:.2f}% {"✅" if p_l2 < 20 else "❌"} (目標 < 20%)')
    print()
    print(f'  平均誤差: {(u_l2 + v_l2) / 2:.2f}% (速度場)')
    print('=' * 70)

In [ ]:
# 5.2.4 顯示視覺化結果
from IPython.display import Image, display
import os

viz_dir = 'results/visualization_re100_kf4/'

if os.path.exists(viz_dir):
    print("📊 場重建對比 (預測 vs 真值 vs 誤差):")
    if os.path.exists(f'{viz_dir}/field_comparison.png'):
        display(Image(filename=f'{viz_dir}/field_comparison.png', width=1200))

    print("\n📈 誤差分佈:")
    if os.path.exists(f'{viz_dir}/error_histograms.png'):
        display(Image(filename=f'{viz_dir}/error_histograms.png', width=800))

    print("\n📊 能譜對比:")
    if os.path.exists(f'{viz_dir}/energy_spectrum_comparison.png'):
        display(Image(filename=f'{viz_dir}/energy_spectrum_comparison.png', width=800))
else:
    print(f"⚠️ 視覺化結果不存在: {viz_dir}")
    print("請先執行上一個 Cell 生成圖表")

### 5.3 訓練失敗診斷

In [ ]:
# 5.3.1 診斷訓練失敗（NaN、損失爆炸等）
!python scripts/debug/diagnose_piratenet_failure.py \
  --checkpoint checkpoints/kolmogorov_re100_kf4_K100/epoch_100.pth \
  --config configs/kolmogorov_re100_kf4_K100.yml \
  --output results/diagnosis/

# 診斷內容：
# - 檢查點完整性分析
# - 損失項趨勢圖（識別發散點）
# - 配置參數驗證
# - 建議修正方案

In [ ]:
# 5.3.2 檢查感測點品質（若資料損失異常高）
!python scripts/visualize_qr_sensors.py \
  --input data/kolmogorov_qr_sensors_re100_kf4_K100.npz \
  --output results/sensor_diagnosis/ \
  --compare-strategies

# 比較不同策略：
# - QR-Pivot (DEIM)
# - POD
# - 隨機採樣
# - 均勻網格

---

## Part 6: 進階實驗

### 6.1 多雷諾數實驗

In [ ]:
# 6.1.1 規劃多雷諾數實驗
!python scripts/calculate_reynolds_parameters.py \
  --f0 1.0 --k 4 --nu-range 0.005 0.025 0.005

# 輸出：
# ν        Re      流動狀態
# 0.005    251.3   湍流
# 0.010    125.7   過渡/弱湍流
# 0.015    83.8    過渡/弱湍流
# 0.020    62.8    過渡/弱湍流
# 0.025    50.3    過渡/弱湍流

In [ ]:
# 6.1.2 批次生成多 Re DNS 資料
re_configs = [
    {'nu': 0.0125, 'kf': 8, 're': 56},
    {'nu': 0.0125, 'kf': 4, 're': 158},
    {'nu': 0.01, 'kf': 4, 're': 197}
]

for cfg in re_configs:
    output_file = f"data/kolmogorov_dns_re{cfg['re']}_512x512_kf{cfg['kf']}.h5"
    if not os.path.exists(output_file):
        print(f"\n生成 Re={cfg['re']} DNS 資料...")
        !python scripts/generate_kolmogorov_dns.py \
          --N 512 --nu {cfg['nu']} --k_f {cfg['kf']} --T_end 40.0 \
          --output {output_file}
    else:
        print(f"✅ Re={cfg['re']} DNS 資料已存在")

### 6.2 K-scan 實驗

In [ ]:
# 6.2.1 生成不同 K 值的感測點配置
K_values = [50, 100, 200]

for K in K_values:
    output_file = f"data/kolmogorov_qr_sensors_re100_kf4_K{K}.npz"
    if not os.path.exists(output_file):
        print(f"\n生成 K={K} 感測點...")
        !python scripts/generate_sensors_k500.py \
          --input data/kolmogorov_dns_re100_512x512_kf4.h5 \
          --K {K} --n-modes 50 \
          --output {output_file}
    else:
        print(f"✅ K={K} 感測點已存在")

In [ ]:
# 6.2.2 比較不同 K 值的條件數與能量比例
import matplotlib.pyplot as plt

K_values = [50, 100, 200]
cond_numbers = []
energy_ratios = []

for K in K_values:
    sensors = np.load(f'data/kolmogorov_qr_sensors_re100_kf4_K{K}.npz')
    cond_numbers.append(sensors.get('condition_number', 0))
    energy_ratios.append(sensors.get('energy_ratio', 0))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(K_values, cond_numbers, 'o-', linewidth=2, markersize=8)
ax1.axhline(y=100, color='r', linestyle='--', label='警告閾值 (100)')
ax1.set_xlabel('感測器數量 K')
ax1.set_ylabel('條件數')
ax1.set_title('條件數 vs K')
ax1.legend()
ax1.grid(True)

ax2.plot(K_values, energy_ratios, 's-', linewidth=2, markersize=8, color='green')
ax2.axhline(y=0.95, color='r', linestyle='--', label='目標閾值 (0.95)')
ax2.set_xlabel('感測器數量 K')
ax2.set_ylabel('能量比例')
ax2.set_title('能量比例 vs K')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('results/k_scan_quality_metrics.png', dpi=150)
plt.show()

print("✅ K-scan 品質指標圖已儲存")

### 6.3 課程學習策略

In [ ]:
# 6.3.1 使用課程學習訓練（Re 遞增）
!python scripts/train_curriculum_kolmogorov.py \
  --config configs/kolmogorov_experiments/kolmogorov_2d_curriculum.yml

# 課程階段：
# Stage 1: Re=30 (層流)，1000 epochs
# Stage 2: Re=56 (過渡)，1000 epochs，載入 Stage 1 權重
# Stage 3: Re=100 (湍流)，2000 epochs，載入 Stage 2 權重

---

## 📚 參考資料

### 文檔
- [`A100_DEPLOYMENT_GUIDE.md`](A100_DEPLOYMENT_GUIDE.md) - NVIDIA A100 部署完整指南
- [`KOLMOGOROV_REYNOLDS_FINAL_REPORT.md`](KOLMOGOROV_REYNOLDS_FINAL_REPORT.md) - Reynolds 修正報告
- [`docs/KOLMOGOROV_DNS_GUIDE.md`](docs/KOLMOGOROV_DNS_GUIDE.md) - DNS 生成完整指南
- [`scripts/README_REYNOLDS_CALCULATOR.md`](scripts/README_REYNOLDS_CALCULATOR.md) - 雷諾數計算器詳細說明
- [`docs/QR_SENSOR_VISUALIZATION_GUIDE.md`](docs/QR_SENSOR_VISUALIZATION_GUIDE.md) - 感測器視覺化指南

### 關鍵腳本
- `scripts/calculate_reynolds_parameters.py` - Reynolds 數計算與驗證 ⭐
- `scripts/generate_kolmogorov_dns.py` - DNS 資料生成
- `scripts/generate_sensors_k500.py` - QR-Pivot 感測器選擇
- `scripts/train.py` - 主訓練腳本
- `scripts/evaluate_checkpoint.py` - 檢查點評估
- `scripts/visualize_kolmogorov_results.py` - 訓練結果視覺化（.npz 檔案）
- `scripts/visualize_results.py` - 模型檢查點視覺化（完整評估）⭐

### 文獻
- Musacchio & Boffetta (2014), *Phys. Rev. E*, 89(2), 023004 - Kolmogorov Flow Reynolds 數定義
- Raissi et al. (2019), *J. Comput. Phys.*, 378, 686-707 - Physics-Informed Neural Networks
- Wang et al. (2021), *J. Comput. Phys.*, 449, 110768 - VS-PINN 變數縮放

---

## 🎓 致謝

本專案使用以下開源工具與資源：
- PyTorch - 深度學習框架
- NumPy & SciPy - 科學計算
- Matplotlib - 視覺化
- h5py - HDF5 資料處理

感謝 Musacchio & Boffetta 提供 Kolmogorov Flow 的標準 Reynolds 數定義。

---

**專案倉庫**: https://github.com/latteine1217/pinns-mvp  
**授權**: MIT License